# Notebook Pruebas

In [1]:
from matplotlib import pyplot as plt
import matplotlib.patches as patches
from matplotlib.animation import FuncAnimation
import trackpy as tp
from skimage import io
import pandas as pd
import pims
import numpy as np
from IPython.display import HTML # For showing the video in Jupyter Notebook


## Archivos

In [2]:
# .tif
PATH_2FPS = "2fps.tif"
PATH_10FPS = "10fps.tif"
PATH_40FPS = "40fps.tif"

STARTING_FRAME = 0
CUT_TOP = 0

frames = pims.open(PATH_10FPS)

In [3]:
frames.shape

(1016, 1024, 504)

## Preprocesado

In [ ]:
# Convertimos los datos a flotante para no perder precisión en la matemática
frames = np.array(frames, dtype=np.float32)
frames = frames[STARTING_FRAME:]
frames[:, :CUT_TOP, :] = 0  

print("frames shape:", frames.shape)

# Visualizamos un frame de ejemplo (el índice 1)
plt.figure(figsize=(6, 8))
plt.imshow(frames[1], cmap="gray")
plt.title("Paso 1: Original Frame (float32)")
plt.axis("off")
plt.show()

Como nuestro sistema es un flujo denso, usamos el máximo temporal (`np.max`) para encontrar el estado donde cada píxel recibió la mayor cantidad de luz (el silo vacío).

In [ ]:
# Buscamos el estado más iluminado de cada píxel en el tiempo
background = np.max(frames, axis=0)
plt.figure(figsize=(6, 8))
plt.imshow(background, cmap="gray")
plt.title("Paso 2: Background Calculado (np.max)")
plt.axis("off")
plt.show()


Restamos el frame actual al fondo. Como las bolitas trazadoras son oscuras, esta operación matemática las invierte y las convierte en puntos de alta intensidad (positivos).

In [ ]:
# Fondo claro - Bolita oscura = Bolita brillante
# np.subtract con out= evita el array temporal que crearía background - frames
np.subtract(background, frames, out=frames)

plt.figure(figsize=(6, 8))
plt.imshow(frames[1], cmap="gray")
plt.title("Paso 3: Sustracción Cruda (Sin contraste)")
plt.axis("off")
plt.show()

Para maximizar la precisión sub-píxel de Trackpy, estiramos el rango dinámico de la imagen para que las trazadoras sean lo más brillantes posible sin llegar a saturarse y perder su forma gaussiana.

In [ ]:
# Encontramos los límites reales de la imagen, ignorando picos extremos de ruido
v_min = np.min(frames)
v_max = np.percentile(frames, 99.9)

# Estiramiento lineal en sitio: modifica frames directamente para no duplicar memoria
frames -= v_min
frames /= (v_max - v_min)
frames *= 240.0
np.clip(frames, 0, 255, out=frames)

# Conversión a uint8 (1 byte/px) y liberación del float32 (4 bytes/px)
frames_clean = frames.astype(np.uint8)
del frames

plt.figure(figsize=(6, 8))
plt.imshow(frames_clean[100], cmap="gray")
plt.title("Paso 4: Frame Limpio y Contrastado")
plt.axis("off")
plt.show()

In [ ]:
from scipy.ndimage import binary_dilation

# 1. Encontramos brillos mayores al umbral extremo (reflejos puros)
led_threshold = 245
led_mask = background > led_threshold
del background  # ya no se usa después de aquí

# ======================================================

# 2. Engordamos un poquito esa máscara para cubrir el resplandor
dilated_led_mask = binary_dilation(led_mask, iterations=3)

# 3. Aplicamos la máscara en sitio ([:, mask] itera sobre todos los frames)
frames_clean[:, dilated_led_mask] = 0
frames_masked = frames_clean
del frames_clean

# Visualizamos los resultados
plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1)
plt.imshow(dilated_led_mask, cmap='gray')
plt.title("Máscara por Umbral Extremo (Brillo > 245)")
plt.axis("off")

plt.subplot(1, 2, 2)
plt.imshow(frames_masked[1], cmap='gray')
plt.title("Frame Limpio y Seguro")
plt.axis("off")

plt.tight_layout()
plt.show()

## Tracking / trackpy

In [ ]:
# 1. Parámetros de prueba (Nuevosjoaco)
# Bajamos diámetro a 7 (para que calce mejor) 
# Bajamos minmass de 500 a 100 o 200 (para atrapar las del centro)
DIAMETER = 9 
MINMASS = 450

# 2. Corremos locate en un solo frame (rápido para iterar)
test_features = tp.locate(frames_masked[100], DIAMETER, minmass=MINMASS)

# 3. Visualizamos con zoom
plt.figure(figsize=(18, 18))
tp.annotate(test_features, frames_masked[100])

In [ ]:
tp.quiet()

features = tp.batch(frames_masked, diameter=DIAMETER, minmass=MINMASS) 
print(len(features), "features encontrados en toda la secuencia")

predictor = tp.predict.NearestVelocityPredict()
t_unfiltered = predictor.link_df(features, search_range=10, memory=4) 
# t = tp.link(features, search_range=10, memory=4) # Ajustados link/memory para flujo denso
print("distinct particles with trajectory tracked (with stubs and low movement particles): ", t_unfiltered.particle.nunique())


In [ ]:
t = tp.filter_stubs(t_unfiltered, 10) # Trayectorias de al menos 10 frames

MINIMUM_Y = 20

particles_delta = t.groupby("particle").aggregate({"y": ["max", "min"]})
particles_with_enough_y_movement = particles_delta[particles_delta[("y", "max")] - particles_delta[("y", "min")] > MINIMUM_Y]
t = t[t.particle.isin(particles_with_enough_y_movement.index)]

# Plotting on the subtracted frame for context
plt.figure(figsize=(12, 12))
tp.plot_traj(t[t["particle"] == 6])
print("distinct particles with trajectory tracked: ", t.particle.nunique())

## Campo de velocidades 

### Calculo 

In [ ]:
t = t.reset_index(drop=True).copy()

Ordenamos por particula y por tiempo (frame) para garantizar la diferencia entre particulas.  

In [ ]:
t.sort_values(['particle', 'frame'], inplace=True)


Luego utilizamos el Filtro de `Savtizky-Golay` en el que en lugar de dos puntos, tomamos una ventana de tiempo y ajustamos un polinomio `p(t) = at^2 + bt + c`. La velocidad es la derivada analitica de ese polinomio. 

In [ ]:
from scipy.signal import savgol_filter

# window_length: Cantidad de frames a mirar para suavizar 
# polyorder: Grado del polinomio a ajustar 
sg_window = 7
sg_poly = 2

def calculate_smooth_velocity(trayectory):
    if len(trayectory) > sg_window:
        # deriv=1 calcula directamente la primera derivada (velocidad en px/frame)
        return savgol_filter(trayectory, window_length=sg_window, polyorder=sg_poly, deriv=1)
    else:
        return np.nan


t['vx'] = t.groupby('particle', group_keys=True).x.transform(calculate_smooth_velocity)
t['vy'] = t.groupby('particle', group_keys=True).y.transform(calculate_smooth_velocity)

t.dropna(subset=['vx', 'vy'], inplace=True)

### Calculo area con velocidad intermedia

In [ ]:
t['v'] = np.sqrt(t['vx']**2 + t['vy']**2)


In [ ]:
t.groupby('particle').agg({'v': ['median', 'std', 'min', 'max', ('q25', lambda x: x.quantile(0.25)), ('q75', lambda x: x.quantile(0.75))]})['v'].describe()

In [ ]:
t.v.describe()
import seaborn as sns
import matplotlib.pyplot as plt

t['v_bin'] = t.v.map(lambda x: round(x, -1))
speed_bins = t[t.v_bin > 0.2].groupby('v_bin').agg({'v': 'count'}).copy()
speed_bins.plot(kind='bar', figsize=(12, 6), title='Distribución de Velocidades (px/frame)', ylabel='Cantidad de Partículas')

In [ ]:
t.particle.nunique()

Lo dividimos en una grilla 

In [ ]:
grid_size = 25 # píxeles
x_bins = np.arange(t.x.min(), t.x.max() + grid_size, grid_size)
y_bins = np.arange(t.y.min(), t.y.max() + grid_size, grid_size)
x_bins

In [ ]:
y_bins

Asignamos por bins: 

In [ ]:
t['x_bin'] = pd.cut(t['x'], bins=x_bins, labels=False)
t['y_bin'] = pd.cut(t['y'], bins=y_bins, labels=False)

nx_bins = len(x_bins) - 1
ny_bins = len(y_bins) - 1

Calculamos las matrices por X e Y: 

In [ ]:
grid_data = t.groupby(['frame', 'y_bin', 'x_bin'])[['vx', 'vy']].mean()

def get_dense_grid(frame_idx, value_col):
    try:
        data = grid_data.loc[frame_idx][value_col].unstack(fill_value=0)
        return data.reindex(index=range(ny_bins), columns=range(nx_bins), fill_value=0).values
    except KeyError:
        return np.zeros((ny_bins, nx_bins))
    
frames_list = sorted(t['frame'].unique())

num_frames = len(frames_list)

Px = np.array([get_dense_grid(f, 'vx') for f in frames_list])
Py = np.array([get_dense_grid(f, 'vy') for f in frames_list])


### Animación de campo: 

In [ ]:
# tomamos el modulo de la velocidad para completar el tercer canal del video
all_speed = np.sqrt(Px**2 + Py**2)

In [ ]:
plt.rcParams['animation.embed_limit'] = 200
fig, ax = plt.subplots(figsize=(10, 8))
X, Y = np.meshgrid(np.arange(nx_bins), np.arange(ny_bins))

Q = ax.quiver(X, Y, Px[0], -Py[0], all_speed[0], cmap="plasma", scale=40, width=0.005)
ax.invert_yaxis()
title = ax.set_title(f"Frame {frames_list[0]}")

def update(i):
    vx, vy, speed = Px[i], Py[i], all_speed[i]
    Q.set_UVC(vx, -vy, speed)
    title.set_text(f"Frame {frames_list[i]}")
    return Q, title

ani = FuncAnimation(fig, update, frames=num_frames, interval=50, blit=True)
ani.save('my_animation.mp4', writer='ffmpeg', fps=20)
plt.close()
HTML(ani.to_jshtml())